# tox21 practice

## Icebreak Rdkit

In [1]:
!pip install rdkit -q

In [11]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd laidd-2026
!ls

Cloning into 'laidd-2026'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 22 (delta 3), reused 8 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 11.61 KiB | 5.81 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/laidd-2026
data  docs  models  notebooks  outputs	README.md  references  src


In [2]:
!mkdir -p src/tools models/tox21_classifier
!ls src

tools


In [3]:
%%writefile src/tools/data_prep.py
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split

TOX21_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def _is_valid_smiles(smiles):
    return Chem.MolFromSmiles(smiles) is not None

def _smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol)
    return np.array(fp)

def load_tox21_clean(test_size=0.3, valid_ratio=0.5, random_state=42):
    df = pd.read_csv(TOX21_URL)
    df['valid'] = df['smiles'].apply(_is_valid_smiles)

    n_total, n_valid = len(df), df['valid'].sum()
    print(f"전체: {n_total}개, 파싱 성공: {n_valid}개, 파싱 실패(제외): {n_total - n_valid}개")

    invalid_smiles = df[~df['valid']]['smiles'].tolist()
    df_clean = df[df['valid']].reset_index(drop=True)

    task_cols = [c for c in df.columns if c not in ['smiles', 'mol_id', 'valid']]

    y = df_clean[task_cols].fillna(0).values.astype(np.float32)
    w = (~df_clean[task_cols].isna()).values.astype(np.float32)
    X = np.stack(df_clean['smiles'].apply(_smiles_to_ecfp).values)

    indices = np.arange(len(X))
    train_idx, temp_idx = train_test_split(indices, test_size=test_size, random_state=random_state)
    valid_idx, test_idx = train_test_split(temp_idx, test_size=valid_ratio, random_state=random_state)

    return {
        'X_train': X[train_idx], 'y_train': y[train_idx], 'w_train': w[train_idx],
        'X_valid': X[valid_idx], 'y_valid': y[valid_idx], 'w_valid': w[valid_idx],
        'X_test': X[test_idx], 'y_test': y[test_idx], 'w_test': w[test_idx],
        'task_cols': task_cols,
        'invalid_smiles': invalid_smiles,
    }

Writing src/tools/data_prep.py


In [4]:
from src.tools.data_prep import load_tox21_clean

data = load_tox21_clean()
X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
X_valid, y_valid, w_valid = data['X_valid'], data['y_valid'], data['w_valid']
X_test, y_test, w_test = data['X_test'], data['y_test'], data['w_test']
task_cols = data['task_cols']

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)
print("Tasks:", task_cols)

[06:24:10] WARNING: not removing hydrogen atom without neighbors
[06:24:11] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:24:11] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:24:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:24:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:24:12] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:24:12] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:24:12] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:24:12] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:24:13] WARNING: not removing hydrogen atom without neighbors


Train: (5476, 2048) Valid: (1173, 2048) Test: (1174, 2048)
Tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

auc_scores = {}
classifiers = {}

for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    valid_mask = w_valid[:, i] == 1

    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])

    probs = clf.predict_proba(X_valid[valid_mask])[:, 1]
    auc = roc_auc_score(y_valid[valid_mask, i], probs)
    auc_scores[task] = auc
    classifiers[task] = clf
    print(f"{task}: AUROC = {auc:.3f}")

print(f"\n평균 AUROC: {np.mean(list(auc_scores.values())):.3f}")

NR-AR: AUROC = 0.820
NR-AR-LBD: AUROC = 0.856
NR-AhR: AUROC = 0.893
NR-Aromatase: AUROC = 0.827
NR-ER: AUROC = 0.748
NR-ER-LBD: AUROC = 0.783
NR-PPAR-gamma: AUROC = 0.847
SR-ARE: AUROC = 0.799
SR-ATAD5: AUROC = 0.869
SR-HSE: AUROC = 0.739
SR-MMP: AUROC = 0.847
SR-p53: AUROC = 0.823

평균 AUROC: 0.821


In [6]:
import joblib
import json
from datetime import datetime

joblib.dump(classifiers, 'models/tox21_classifier/rf_baseline.pkl')

metadata = {
    "model_type": "RandomForestClassifier",
    "featurizer": "ECFP (radius=2, n_bits=2048)",
    "class_weight": "balanced",
    "auc_scores": auc_scores,
    "mean_auc": float(np.mean(list(auc_scores.values()))),
    "created_at": datetime.now().isoformat(),
}
with open('models/tox21_classifier/rf_baseline_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("저장 완료")

저장 완료


In [7]:
!ls -la models/tox21_classifier/
!cat models/tox21_classifier/rf_baseline_meta.json

total 106580
drwxr-xr-x 2 root root      4096 Jul 17 06:31 .
drwxr-xr-x 3 root root      4096 Jul 17 06:24 ..
-rw-r--r-- 1 root root       649 Jul 17 06:31 rf_baseline_meta.json
-rw-r--r-- 1 root root 109124410 Jul 17 06:31 rf_baseline.pkl
{
  "model_type": "RandomForestClassifier",
  "featurizer": "ECFP (radius=2, n_bits=2048)",
  "class_weight": "balanced",
  "auc_scores": {
    "NR-AR": 0.8199715510599206,
    "NR-AR-LBD": 0.8555570398076409,
    "NR-AhR": 0.8932149633798685,
    "NR-Aromatase": 0.8268479567307694,
    "NR-ER": 0.7476891152612715,
    "NR-ER-LBD": 0.7834293418311067,
    "NR-PPAR-gamma": 0.846767893760375,
    "SR-ARE": 0.7986716106271149,
    "SR-ATAD5": 0.868685240617085,
    "SR-HSE": 0.7391053152855728,
    "SR-MMP": 0.8472354417236307,
    "SR-p53": 0.8225976589083979
  },
  "mean_auc": 0.8208144274160628,
  "created_at": "2026-07-17T06:31:58.546491"
}

In [12]:
!mkdir -p /content/laidd-2026/models/tox21_classifier
!mv /content/models/tox21_classifier/rf_baseline_meta.json /content/laidd-2026/models/tox21_classifier/
!mv /content/models/tox21_classifier/rf_baseline.pkl /content/laidd-2026/models/tox21_classifier/

In [13]:
%cd /content/laidd-2026
!ls models/tox21_classifier/
!git status

/content/laidd-2026
rf_baseline_meta.json  rf_baseline.pkl
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/tox21_classifier/
	src/tools/

nothing added to commit but untracked files present (use "git add" to track)


In [14]:
!git add src/tools/data_prep.py models/tox21_classifier/rf_baseline_meta.json
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   models/tox21_classifier/rf_baseline_meta.json
	new file:   src/tools/data_prep.py



In [15]:
!git config --global user.email "hkw_61@naver.com"
!git config --global user.name "Dec32th"
!git commit -m "Add data prep function and baseline RF model results (mean AUROC 0.821)"

[main f77f1d4] Add data prep function and baseline RF model results (mean AUROC 0.821)
 2 files changed, 66 insertions(+)
 create mode 100644 models/tox21_classifier/rf_baseline_meta.json
 create mode 100644 src/tools/data_prep.py
fatal: could not read Password for 'https://%7Btoken%7D@github.com': No such device or address


In [17]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git push https://{token}@github.com/Dec32th/laidd-2026.git

Enumerating objects: 11, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (8/8), 1.75 KiB | 1.75 MiB/s, done.
Total 8 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Dec32th/laidd-2026.git
   8fc5d8b..f77f1d4  main -> main
